In [ ]:
# Actividad Semana 10
**Alumno:** Fernanda Olvera  
**Materia:** Módulo 3 – IA aplicada  
**Objetivo:** Entrenar y comparar modelos de clasificación y regresión para predicción de riesgo y recuperación esperada.

In [ ]:
import pandas as pd

df = pd.read_csv("dataset_predictive_collection.csv")

print("=== Primeras filas del dataset ===")
print(df.head())
print("\n=== Estadísticas descriptivas ===")
print(df.describe())


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

X_class = df[["monto_factura","dias_atraso","historial_pago","frecuencia_disputas"]]
y_class = df["riesgo"]

le = LabelEncoder()
y_class_encoded = le.fit_transform(y_class)

X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(
    X_class, y_class_encoded, test_size=0.2, random_state=42, stratify=y_class_encoded
)

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_class, y_train_class)
y_pred_class = log_reg.predict(X_test_class)

acc = accuracy_score(y_test_class, y_pred_class)
f1 = f1_score(y_test_class, y_pred_class, average="weighted")

print("Accuracy:", acc)
print("F1-Score:", f1)
print("\nReporte de clasificación:\n", classification_report(y_test_class, y_pred_class))

# Matriz de confusión
cm = confusion_matrix(y_test_class, y_pred_class)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Matriz de Confusión - Clasificación")
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.show()

joblib.dump(log_reg, "classification_model.pkl")
joblib.dump(le, "label_encoder.pkl")


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score

X_reg = df[["monto_factura","dias_atraso","historial_pago","frecuencia_disputas"]]
y_reg = df["recuperacion_esperada"]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

lin_reg = LinearRegression()
lin_reg.fit(X_train_reg, y_train_reg)
y_pred_reg = lin_reg.predict(X_test_reg)

mse = mean_squared_error(y_test_reg, y_pred_reg)
rmse = mse**0.5
r2 = r2_score(y_test_reg, y_pred_reg)

print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.4f}")

cv_scores = cross_val_score(lin_reg, X_reg, y_reg, cv=5, scoring="r2")
print(f"Cross-Validation R² promedio: {cv_scores.mean():.4f}")

# Gráfica de dispersión
plt.figure(figsize=(6,6))
plt.scatter(y_test_reg, y_pred_reg, alpha=0.6)
plt.xlabel("Valores reales")
plt.ylabel("Predicciones")
plt.title("Regresión Lineal - Valores reales vs predichos")
plt.plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], "r--")
plt.show()

# Distribución de errores
errores = y_test_reg - y_pred_reg
sns.histplot(errores, bins=20, kde=True)
plt.title("Distribución de errores (residuos)")
plt.xlabel("Error")
plt.ylabel("Frecuencia")
plt.show()

joblib.dump(lin_reg, "regression_model.pkl")


In [ ]:
comparacion = pd.DataFrame({
    "Modelo": ["Clasificación (Logistic Regression)", "Regresión Lineal"],
    "Métrica Principal": ["F1-Score", "R²"],
    "Valor": [f1, r2]
})

print("=== Comparación Final ===")
print(comparacion)

comparacion.to_csv("comparacion_modelos.csv", index=False)
